In [13]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import optuna
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor
from Preprocess import preprocess_data_window
from catboost import CatBoostRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [14]:
# ── Cell 2: Load data ────────────────────────────────────────────────────────
data_orig     = pd.read_csv("../Whillians-GPS-Data-and-Features.csv")
filtered_time = pd.read_csv("../filtered_time_to_next_event.csv")

In [15]:
# ── Cell 3: Preprocessing ────────────────────────────────────────────────────
n_previous_events = 20
n_qubits_base = 6  # features per event

X_train, X_val, X_test, y_train, y_val, y_test, feature_cols = preprocess_data_window(
    filtered_time, data_orig, n_previous_events
)

print("NaN count per column:\n", X_train.isna().sum())
print("Total NaN:", X_train.isna().sum().sum())

import pandas as pd
print(pd.Series(y_train).describe())
print("\nSkew:", pd.Series(y_train).skew())

X shape:  (2822, 126)
y shape:  (2822,)
NaN count per column:
 tide_deriv-0      0
form_fac-0        0
time_since-0      0
slip_size-0       0
high_t_evt-0      0
                 ..
form_fac-20       0
time_since-20     0
slip_size-20      0
high_t_evt-20     0
tide_height-20    0
Length: 126, dtype: int64
Total NaN: 0
count      1692.000000
mean      60206.888298
std       20347.145025
min       23265.000000
25%       43316.250000
50%       54832.500000
75%       82443.750000
max      119220.000000
Name: time_to_next_ev_hr, dtype: float64

Skew: 0.27697104661890737


In [16]:
# ── Cell 4: Config ───────────────────────────────────────────────────────────
QRC_CONFIG = {
    "num_layers_per_event": 3,
    "shots": 5000,
    "n_iterations": 10,
    "top_k": 5,
    "correlation_threshold": 0.0,
    "random_seed": 42,
}

CLASSICAL_MODEL_NAME = "XGBoost"
n_previous_events = 20

print("QRC Config:", QRC_CONFIG)
print("Classical model:", CLASSICAL_MODEL_NAME)

QRC Config: {'num_layers_per_event': 3, 'shots': 5000, 'n_iterations': 10, 'top_k': 5, 'correlation_threshold': 0.0, 'random_seed': 42}
Classical model: XGBoost


In [17]:
# ── Cell 5+6: Scaling only ────────────────────────────────────────────────────
def scale_to_pi_range(X_train, X_val, X_test):
    train_min = X_train.min(axis=0)
    train_max = X_train.max(axis=0)
    denom = train_max - train_min
    denom[denom == 0] = 1.0

    def transform(X):
        scaled = (X - train_min) / denom
        scaled = np.clip(scaled, 0.0, 1.0)
        return scaled * np.pi

    return transform(X_train), transform(X_val), transform(X_test)


X_train_sel = X_train.to_numpy()
X_val_sel   = X_val.to_numpy()
X_test_sel  = X_test.to_numpy()

X_train_q, X_val_q, X_test_q = scale_to_pi_range(X_train_sel, X_val_sel, X_test_sel)

n_qubits       = n_qubits_base
n_states       = 2 ** n_qubits
n_total_events = n_previous_events + 1

print(f"Input to quantum circuit: {X_train_q.shape}")
print(f"n_qubits: {n_qubits}, n_states: {n_states}")
print(f"Total events per sample: {n_total_events}")

Input to quantum circuit: (1692, 126)
n_qubits: 6, n_states: 64
Total events per sample: 21


In [18]:
# ── Cell 7: FC-TFI with tuned Hamiltonian parameters ─────────────────────────
def generate_ising_params(n_qubits, rng, J_std=1.0, h=1.5, t=0.8):
    """
    Generate FC-TFI Hamiltonian parameters.

    Key changes from previous run:
      J_std: 0.5 → 1.0 — stronger disorder in couplings improves
             memory capacity (Koskela et al. 2020 shows higher J_std
             gives better separation of input states)
      h:     1.0 → 1.5 — stronger transverse field pushes system
             closer to quantum critical point where entanglement and
             information spreading are maximized
      t:     0.5 → 0.8 — longer evolution time allows more information
             to spread across qubits before measurement, but not so
             long that information is lost to scrambling
    """
    J = np.zeros((n_qubits, n_qubits))
    for i in range(n_qubits):
        for j in range(i + 1, n_qubits):
            J[i, j] = rng.normal(0, J_std)
    return J, h, t


def trotter_ising_layer(qc, n_qubits, J, h, t, n_trotter_steps=4):
    """
    Trotter decomposition of FC-TFI Hamiltonian.
    Increased from 3 to 4 steps for better accuracy at t=0.8.
    """
    dt = t / n_trotter_steps

    for _ in range(n_trotter_steps):
        for i in range(n_qubits):
            for j in range(i + 1, n_qubits):
                if abs(J[i, j]) > 1e-10:
                    qc.cx(i, j)
                    qc.rz(2 * J[i, j] * dt, j)
                    qc.cx(i, j)
        for i in range(n_qubits):
            qc.rx(2 * h * dt, i)


def encode_rotations(qc, data_sample, n_qubits):
    for i in range(n_qubits):
        qc.ry(float(data_sample[i]), i)


def build_reservoir_circuit_rotations(data_sample, ising_params, num_layers, n_qubits):
    """
    FC-TFI reservoir with data re-uploading.
    Increased Trotter accuracy and tuned Hamiltonian parameters.
    """
    J, h, t = ising_params
    qc = QuantumCircuit(n_qubits)

    for i in range(n_qubits):
        qc.h(i)

    encode_rotations(qc, data_sample, n_qubits)
    qc.barrier()

    for layer in range(num_layers):
        trotter_ising_layer(qc, n_qubits, J, h, t)
        qc.barrier()

        encode_rotations(qc, data_sample, n_qubits)
        qc.barrier()

        trotter_ising_layer(qc, n_qubits, J, h, t)
        qc.barrier()

    return qc


def generate_random_angles(num_layers, n_qubits, rng):
    """Returns Ising params tuple for API compatibility."""
    return generate_ising_params(n_qubits, rng)

In [19]:
# ── Cell 8: Pauli measurement runner — FC-TFI compatible ─────────────────────
def get_pauli_expectations(circuit_base, n_qubits, shots, simulator):
    """
    Measure symmetry-aware Pauli observables.

    For the FC-TFI Hamiltonian with ZZ + X structure, the natural
    symmetry-aware observables are:
      - Total magnetization: <Z_0 + Z_1 + ... + Z_{n-1}> / n
      - Nearest-neighbor ZZ correlations: <Z_i Z_{i+1}>
      - Total X magnetization: <X_0 + ... + X_{n-1}> / n
      - Individual qubit expectations for readout richness

    We measure all individual X, Y, Z expectations (18 values) plus
    two-qubit ZZ correlators for nearest neighbors (6 values for ring)
    giving 24 total features per circuit — richer than plain Pauli
    and aligned with Hamiltonian symmetry structure.

    Returns shape (3 * n_qubits + n_qubits,) = (24,) for 6 qubits.
    """
    expectations = np.zeros(3 * n_qubits)

    # Single-qubit Pauli expectations
    for basis_idx, basis in enumerate(['Z', 'X', 'Y']):
        qc_meas = circuit_base.copy()

        if basis == 'X':
            for i in range(n_qubits):
                qc_meas.h(i)
        elif basis == 'Y':
            for i in range(n_qubits):
                qc_meas.sdg(i)
                qc_meas.h(i)

        qc_meas.measure_all()
        result = simulator.run(qc_meas, shots=shots).result()
        counts = result.get_counts()

        for bitstring, count in counts.items():
            bits = bitstring.replace(" ", "")
            for qubit_idx in range(n_qubits):
                bit = int(bits[n_qubits - 1 - qubit_idx])
                expectations[basis_idx * n_qubits + qubit_idx] += (
                    (1 - 2 * bit) * count / shots
                )

    # ZZ correlators — symmetry-aware, suppresses concentration
    # Measure <Z_i Z_{i+1}> for ring topology (matches CNOT entanglement)
    zz_correlators = np.zeros(n_qubits)
    qc_zz = circuit_base.copy()
    qc_zz.measure_all()
    result_zz = simulator.run(qc_zz, shots=shots).result()
    counts_zz = result_zz.get_counts()

    for bitstring, count in counts_zz.items():
        bits = bitstring.replace(" ", "")
        for i in range(n_qubits):
            j = (i + 1) % n_qubits  # ring topology
            bit_i = int(bits[n_qubits - 1 - i])
            bit_j = int(bits[n_qubits - 1 - j])
            zi = 1 - 2 * bit_i
            zj = 1 - 2 * bit_j
            zz_correlators[i] += zi * zj * count / shots

    return np.concatenate([expectations, zz_correlators])  # shape (24,)


def run_quantum_reservoir_pauli(
    X_data, angle_bank, num_layers, n_qubits, n_total_events, shots
):
    """
    FC-TFI Pauli reservoir runner.
    Output shape: (m, n_total_events * 24) for 6 qubits.
    angle_bank contains Ising param tuples (J, h, t) per event slot.
    """
    if hasattr(X_data, 'values'):
        X_data = X_data.to_numpy()

    m          = X_data.shape[0]
    n_obs      = 3 * n_qubits + n_qubits  # 24 for 6 qubits
    simulator  = AerSimulator()
    pauli_matrix = np.zeros((m, n_total_events * n_obs))

    for event_idx in range(n_total_events):
        start_col    = event_idx * n_qubits
        end_col      = start_col + n_qubits
        X_event      = X_data[:, start_col:end_col]
        ising_params = angle_bank[event_idx]  # (J, h, t) tuple

        for sample_idx in range(m):
            qc = build_reservoir_circuit_rotations(
                X_event[sample_idx], ising_params, num_layers, n_qubits
            )
            obs_vals = get_pauli_expectations(qc, n_qubits, shots, simulator)
            pauli_matrix[
                sample_idx,
                event_idx * n_obs:(event_idx + 1) * n_obs
            ] = obs_vals

        print(f"  Event {event_idx + 1}/{n_total_events} done", end="\r")

    print()
    return pauli_matrix


def make_hybrid_features(P, X_classical):
    return np.hstack([P, X_classical])


def make_hybrid_features_decay(P_matrix, X_classical, n_total_events, n_obs, decay=0.3):
    """
    Exponential decay weighting on FC-TFI Pauli features.
    Most recent event weighted highest, oldest weighted lowest.
    """
    weights = np.array([
        np.exp(-decay * (n_total_events - 1 - i))
        for i in range(n_total_events)
    ])
    weights /= weights.sum()

    weighted_P = P_matrix.copy()
    for event_idx in range(n_total_events):
        start = event_idx * n_obs
        end   = start + n_obs
        weighted_P[:, start:end] *= weights[event_idx]

    return np.hstack([weighted_P, X_classical])

In [20]:
# ── Cell 9: Model registry ───────────────────────────────────────────────────
def get_model_registry():
    return {
        "Ridge": {
            "optuna_fn": lambda trial: {
                "alpha": trial.suggest_float("alpha", 1e-2, 1e8, log=True),
            },
            "fixed_params": {},
            "build_fn": lambda params: Ridge(**params),
            "fit_fn": lambda model, Xt, yt, Xv, yv: model.fit(Xt, yt),
            "shap_explainer": "linear",
        },
        "XGBoost": {
            "optuna_fn": lambda trial: {
                "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
                "max_depth":        trial.suggest_int("max_depth", 2, 6),
                "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
                "min_child_weight": trial.suggest_int("min_child_weight", 3, 20),
                # Push harder on regularization — 1470-dim input needs it
                "reg_alpha":        trial.suggest_float("reg_alpha", 0.1, 50.0, log=True),
                "reg_lambda":       trial.suggest_float("reg_lambda", 0.1, 50.0, log=True),
                # Limit tree complexity on high-dim noisy input
                "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
                "max_delta_step":   trial.suggest_int("max_delta_step", 0, 5),
            },
            "fixed_params": {
                "objective":             "reg:squarederror",
                "n_estimators":          2000,
                "random_state":          42,
                "early_stopping_rounds": 75,  # more patience on larger feature set
                "tree_method":           "hist",  # faster on wide feature matrices
            },
            "build_fn": lambda params: XGBRegressor(**params),
            "fit_fn": lambda model, Xt, yt, Xv, yv: model.fit(
                Xt, yt, eval_set=[(Xv, yv)], verbose=False
            ),
            "shap_explainer": "tree",
        },
        "CatBoost": {
            "optuna_fn": lambda trial: {
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "depth":         trial.suggest_int("depth", 2, 8),
            },
            "fixed_params": {
                "iterations":          100,
                "loss_function":       "RMSE",
                "eval_metric":         "RMSE",
                "random_seed":         42,
                "verbose":             False,
                "allow_writing_files": False,
            },
            "build_fn": lambda params: CatBoostRegressor(**params),
            "fit_fn": lambda model, Xt, yt, Xv, yv: model.fit(
                Xt, yt, eval_set=(Xv, yv), use_best_model=True
            ),
            "shap_explainer": "tree",
        },
        "RandomForest": {
            "optuna_fn": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
                "max_depth":    trial.suggest_int("max_depth", 2, 10),
            },
            "fixed_params": {"random_state": 42},
            "build_fn": lambda params: RandomForestRegressor(**params),
            "fit_fn": lambda model, Xt, yt, Xv, yv: model.fit(Xt, yt),
            "shap_explainer": "tree",
        },
    }


registry = get_model_registry()
print(f"Available models: {list(registry.keys())}")
print(f"Using: {CLASSICAL_MODEL_NAME}")

Available models: ['Ridge', 'XGBoost', 'CatBoost', 'RandomForest']
Using: XGBoost


In [21]:
# ── Cell 10: Single iteration — FC-TFI ───────────────────────────────────────
def run_single_qrc_iteration(
    iteration_idx,
    X_train_q, X_val_q, X_test_q,
    y_train, y_val, y_test,
    n_qubits, config, model_name, registry,
):
    num_layers     = config["num_layers_per_event"]
    shots          = config["shots"]
    seed           = config["random_seed"] + iteration_idx
    n_total_events = n_previous_events + 1
    n_obs          = 3 * n_qubits + n_qubits  # 24 — ZZ correlators added

    rng = np.random.default_rng(seed)

    # Generate FC-TFI Ising params per event slot
    # Each event gets different disordered couplings J_ij
    # Disorder is key for memory capacity (Koskela et al. 2020)
    angle_bank = [
        generate_random_angles(num_layers, n_qubits, rng)
        for _ in range(n_total_events)
    ]

    print(f"  Running FC-TFI reservoir on train ({X_train_q.shape[0]} samples)...")
    P_train = run_quantum_reservoir_pauli(
        X_train_q, angle_bank, num_layers, n_qubits, n_total_events, shots
    )
    print(f"  Running FC-TFI reservoir on val ({X_val_q.shape[0]} samples)...")
    P_val = run_quantum_reservoir_pauli(
        X_val_q, angle_bank, num_layers, n_qubits, n_total_events, shots
    )
    print(f"  Running FC-TFI reservoir on test ({X_test_q.shape[0]} samples)...")
    P_test = run_quantum_reservoir_pauli(
        X_test_q, angle_bank, num_layers, n_qubits, n_total_events, shots
    )

    # Use exponential decay weighting — physically motivated for temporal data
    H_train = make_hybrid_features_decay(P_train, X_train_q, n_total_events, n_obs)
    H_val   = make_hybrid_features_decay(P_val,   X_val_q,   n_total_events, n_obs)
    H_test  = make_hybrid_features_decay(P_test,  X_test_q,  n_total_events, n_obs)

    entry = registry[model_name]

    def objective(trial):
        tuned  = entry["optuna_fn"](trial)
        params = {**entry["fixed_params"], **tuned}
        model  = entry["build_fn"](params)
        entry["fit_fn"](model, H_train, y_train, H_val, y_val)
        preds  = model.predict(H_val)
        return mean_absolute_error(y_val, preds)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=50)

    best_tuned  = entry["optuna_fn"](study.best_trial)
    best_params = {**entry["fixed_params"], **best_tuned}
    model       = entry["build_fn"](best_params)
    entry["fit_fn"](model, H_train, y_train, H_val, y_val)

    val_pred  = model.predict(H_val)
    test_pred = model.predict(H_test)

    return {
        "iteration":   iteration_idx,
        "val_r2":      r2_score(y_val, val_pred),
        "val_rmse":    root_mean_squared_error(y_val, val_pred),
        "val_mae":     mean_absolute_error(y_val, val_pred),
        "test_rmse":   root_mean_squared_error(y_test, test_pred),
        "test_mae":    mean_absolute_error(y_test, test_pred),
        "test_pred":   test_pred,
        "best_params": best_params,
        "model":       model,
        "P_train": P_train, "P_val": P_val, "P_test": P_test,
    }

In [22]:
# ── Cell 11: Pipeline + run ──────────────────────────────────────────────────
def run_qrc_pipeline(
    X_train_q, X_val_q, X_test_q,
    y_train, y_val, y_test,
    n_qubits, config, model_name, registry,
):
    results = []
    for i in range(config["n_iterations"]):
        print(f"\nIteration {i + 1}/{config['n_iterations']}:")
        res = run_single_qrc_iteration(
            i,
            X_train_q, X_val_q, X_test_q,
            y_train, y_val, y_test,
            n_qubits, config, model_name, registry,
        )
        print(
            f"\tVal R2: {res['val_r2']:.4f} | Val RMSE: {res['val_rmse']:.2f} | "
            f"Val MAE: {res['val_mae']:.2f} | Test RMSE: {res['test_rmse']:.2f} | "
            f"Test MAE: {res['test_mae']:.2f}"
        )
        results.append(res)

    top_k          = config["top_k"]
    sorted_results = sorted(results, key=lambda r: r["val_mae"])
    top_results    = sorted_results[:top_k]
    top_indices    = [r["iteration"] for r in top_results]

    print(f"\nTop-{top_k} iterations (by val MAE): {[i + 1 for i in top_indices]}")

    ensemble_pred = np.mean([r["test_pred"] for r in top_results], axis=0)
    ensemble_rmse = root_mean_squared_error(y_test, ensemble_pred)
    ensemble_mae  = mean_absolute_error(y_test, ensemble_pred)
    ensemble_r2   = r2_score(y_test, ensemble_pred)

    print(f"\nEnsemble Test RMSE: {ensemble_rmse:.2f}")
    print(f"Ensemble Test MAE:  {ensemble_mae:.2f}")
    print(f"Ensemble Test R2:   {ensemble_r2:.4f}")

    return results, ensemble_pred, top_indices


all_results, ensemble_pred, top_indices = run_qrc_pipeline(
    X_train_q,
    X_val_q,
    X_test_q,
    y_train,
    y_val,
    y_test,
    n_qubits,
    QRC_CONFIG,
    "XGBoost",
    registry,
)


Iteration 1/10:
  Running FC-TFI reservoir on train (1692 samples)...
  Event 21/21 done
  Running FC-TFI reservoir on val (565 samples)...
  Event 21/21 done
  Running FC-TFI reservoir on test (565 samples)...
  Event 21/21 done
	Val R2: 0.2924 | Val RMSE: 16877.85 | Val MAE: 13032.63 | Test RMSE: 15548.14 | Test MAE: 12079.22

Iteration 2/10:
  Running FC-TFI reservoir on train (1692 samples)...
  Event 21/21 done
  Running FC-TFI reservoir on val (565 samples)...
  Event 21/21 done
  Running FC-TFI reservoir on test (565 samples)...
  Event 21/21 done


[W 2026-03-24 13:26:39,959] Trial 43 failed with parameters: {'learning_rate': 0.010473676441748235, 'max_depth': 6, 'subsample': 0.9724063311053397, 'colsample_bytree': 0.5017188591582199, 'min_child_weight': 4, 'reg_alpha': 1.8884494642167302, 'reg_lambda': 0.21955563597740207, 'gamma': 0.8404601266773901, 'max_delta_step': 0} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\kaitl\OneDrive\Documents\Icequake Modeling\Code\Icequake-QRC-\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\kaitl\AppData\Local\Temp\ipykernel_44556\2562296977.py", line 48, in objective
    entry["fit_fn"](model, H_train, y_train, H_val, y_val)
  File "C:\Users\kaitl\AppData\Local\Temp\ipykernel_44556\1804726616.py", line 35, in <lambda>
    "fit_fn": lambda model, Xt, yt, Xv, yv: model.fit(
                                            ^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
import pickle
import os

run_label = "Hamil-1-QRC_pauli"  # change per run

save_data = {
    "ensemble_pred": ensemble_pred,
    "y_test": y_test,
    "all_results": [
        {k: v for k, v in r.items() if k != "model"}
        for r in all_results
    ],
    "ensemble_mae": mean_absolute_error(y_test, ensemble_pred),
    "ensemble_rmse": root_mean_squared_error(y_test, ensemble_pred),
    "ensemble_r2": r2_score(y_test, ensemble_pred),
}

os.makedirs("results", exist_ok=True)
with open(f"results/{run_label}.pkl", "wb") as f:
    pickle.dump(save_data, f)

print(f"Saved results to results/{run_label}.pkl")

Saved results to results/Hamil-QRC_pauli.pkl


In [ ]:
# ── Cell 12: Classical baseline diagnostic ───────────────────────────────────
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

last_res = all_results[-1]
P_dim = last_res["P_train"].shape[1]

H_train = np.hstack([last_res["P_train"], X_train_q])
H_val   = np.hstack([last_res["P_val"],   X_val_q])
H_test  = np.hstack([last_res["P_test"],  X_test_q])

X_tr_raw = H_train[:, P_dim:]
X_v_raw  = H_val[:,   P_dim:]
X_te_raw = H_test[:,  P_dim:]

classical_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    early_stopping_rounds=75,
    tree_method="hist",
    random_state=42,
)

classical_model.fit(
    X_tr_raw, y_train,
    eval_set=[(X_v_raw, y_val)],
    verbose=False
)

classical_pred = classical_model.predict(X_te_raw)
classical_mae  = mean_absolute_error(y_test, classical_pred)
quantum_mae    = last_res["test_mae"]

print(f"Classical XGBoost (126 features, no quantum): MAE = {classical_mae:.2f}")
print(f"Quantum hybrid (single iteration):            MAE = {quantum_mae:.2f}")
print(f"Quantum vs classical difference:              {quantum_mae - classical_mae:.2f}s")

Classical XGBoost (126 features, no quantum): MAE = 11671.34
Quantum hybrid (single iteration):            MAE = 11441.80
Quantum vs classical difference:              -229.54s
